In [0]:
storage_account = "dmpro2"
access_key = "blkSylna0g9RsyAQUhzDDyJZ4fS4oWfrveP8HO+FYhp0Y1Cv6hcIGjo2UmS/MWA/TWcTWiAHRTy5+AStQPiZYA=="   # hard-coded

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", access_key)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

catalog_name = "dm_pro2"

products_schema = StructType([
    StructField("ProductID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Price", DoubleType(), True),
    StructField("StockQuantity", IntegerType(), True),
    StructField("Supplier", StringType(), True),
])

raw_data_path = f"abfss://rawdata@dmpro2.dfs.core.windows.net/data/products/*.csv"

df = spark.read.option('header', "true").schema(products_schema).csv(raw_data_path)

# add metadata columns
df = df.withColumn("_source_file", col("_metadata.file_path")) \
       .withColumn("ingested_at", current_timestamp())

#display(df.limit(5))
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.products")

customers_schema = StructType([
    StructField("CustomerID", StringType(), True),
    StructField("FirstName", StringType(), True),
    StructField("LastName", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("Phone", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("LoyaltyPoints", StringType(), True),
])


raw_data_path = f"abfss://rawdata@dmpro2.dfs.core.windows.net/data/customers/*.csv"


df = spark.read.option('header', "true").schema(customers_schema).csv(raw_data_path)


# add metadata columns
df = df.withColumn("_source_file", col("_metadata.file_path")) \
       .withColumn("ingested_at", current_timestamp())


#display(df.limit(5))
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.customers")   
